
## Introduzione e Definizione del Problema

### Contesto applicativo

Il progetto affronta un problema di **Network Intrusion Detection** sul dataset UNSW-NB15. Ogni riga descrive un flusso di rete tramite feature numeriche e categoriche; l'obiettivo è costruire modelli di Machine Learning capaci di distinguere traffico normale e traffico malevolo.

### Task di Machine Learning

In questo progetto consideriamo due possibili formulazioni del problema:

- **Task principale - classificazione binaria:** predire `label`, dove `0` indica traffico normale e `1` indica traffico di attacco.
- **Task secondario/opzionale - classificazione multiclasse:** predire `attack_cat`, cioè la categoria dell'attacco.

La classificazione binaria viene trattata come task principale perché è piu coerente con un primo sistema IDS e risulta meno critica rispetto alla classificazione multiclasse, dove alcune classi di attacco sono fortemente sottorappresentate.

### Protocollo sperimentale

Il dataset fornisce una partizione ufficiale in training set e test set. Per evitare data leakage:

- l'EDA e le decisioni di preprocessing vengono svolte sul **training set**;
- il **test set** viene mantenuto separato e sarà utilizzato solo per la valutazione finale;
- la selezione dei modelli e degli iperparametri sarà svolta tramite cross-validation sul training set.

### Obiettivi dell'EDA

- Comprendere struttura, tipi di variabili e distribuzioni del training set.
- Identificare problemi di qualità dei dati: missing values, duplicati, inconsistenze e valori estremi.
- Analizzare sbilanciamento delle classi e relazioni tra feature e target.
- Derivare decisioni motivate per preprocessing, feature engineering e modellazione.


In [ ]:
from google.colab import drive
import os

# Monta Google Drive
drive.mount('/content/drive')

In [ ]:
# ==========================================
# 2. IMPORT DELLE LIBRERIE E CONFIGURAZIONE
# ==========================================
import os
import re
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import missingno as msno
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, RobustScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from typing import List, Tuple, Dict, Any

# Configurazione stile grafici e opzioni globali
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
sns.set_context("notebook", font_scale=1.1)

# Configurazioni Pandas per dataset larghi
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Disabilitazione warning non critici
warnings.filterwarnings('ignore')

# Riproducibilita'
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Librerie importate e ambiente configurato correttamente.")


### Percorsi, Output e Caricamento del Dataset

In questa sezione definiamo i percorsi principali del progetto e creiamo le cartelle in cui salvare gli artefatti prodotti dall'EDA.

Gli output vengono organizzati in `outputs/eda/`, con sottocartelle dedicate a duplicati, statistiche descrittive, outlier, distribuzioni, correlazioni, target, relazioni tra feature, qualit? dei dati, PCA e feature importance. L'EDA continua a usare solo il **training set ufficiale**; il test set viene solo referenziato e resta riservato alla valutazione finale.


In [ ]:
# ==========================================
# 3. PERCORSI, OUTPUT E CARICAMENTO DATASET
# ==========================================
BASE_PATH = Path("/content/drive/MyDrive/progetto FDSL")
REPO_PATH = BASE_PATH / "unsw-nb15-network-ids"
DATA_PATH = BASE_PATH

TRAIN_PATH = DATA_PATH / "UNSW_NB15_training-set.parquet"
TEST_PATH = DATA_PATH / "UNSW_NB15_testing-set.parquet"

OUTPUTS_PATH = REPO_PATH / "results"
EDA_OUTPUTS_PATH = OUTPUTS_PATH / "eda"

EDA_DIRS = {
    "eda_root": EDA_OUTPUTS_PATH,
    "duplicates": EDA_OUTPUTS_PATH / "duplicates",
    "descriptive_statistics": EDA_OUTPUTS_PATH / "descriptive_statistics",
    "outliers": EDA_OUTPUTS_PATH / "outliers",
    "outlier_boxplots": EDA_OUTPUTS_PATH / "outliers" / "boxplots",
    "distributions": EDA_OUTPUTS_PATH / "distributions",
    "numeric_distributions": EDA_OUTPUTS_PATH / "distributions" / "numeric",
    "categorical_distributions": EDA_OUTPUTS_PATH / "distributions" / "categorical",
    "correlations": EDA_OUTPUTS_PATH / "correlations",
    "target": EDA_OUTPUTS_PATH / "target",
    "feature_relationships": EDA_OUTPUTS_PATH / "feature_relationships",
    "data_quality": EDA_OUTPUTS_PATH / "data_quality",
    "pca_feature_importance": EDA_OUTPUTS_PATH / "pca_feature_importance",
}

TARGET_BINARY = "label"
TARGET_MULTICLASS = "attack_cat"
TARGET_COLUMNS = [TARGET_BINARY, TARGET_MULTICLASS]


def slugify(value: str) -> str:
    """
    Converte nomi di feature/titoli in nomi file sicuri e leggibili.
    """
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9_\-]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"


def ensure_project_directories() -> None:
    """
    Verifica i percorsi principali e crea le cartelle di output EDA.
    """
    print("Verifica percorsi progetto")
    print("=" * 70)

    required_paths = {
        "Base progetto": BASE_PATH,
        "Repository": REPO_PATH,
        "Training set": TRAIN_PATH,
        "Test set": TEST_PATH,
    }

    all_inputs_ok = True
    for path_name, path in required_paths.items():
        if path.exists():
            print(f"OK  {path_name:<18} {path}")
        else:
            print(f"NO  {path_name:<18} {path}")
            all_inputs_ok = False

    print("-" * 70)
    for path_name, path in EDA_DIRS.items():
        path.mkdir(parents=True, exist_ok=True)
        print(f"OUT {path_name:<18} {path}")

    print("=" * 70)
    if all_inputs_ok:
        print("Percorsi di input trovati; cartelle output EDA pronte.")
    else:
        print("ATTENZIONE: uno o piu' percorsi di input non sono stati trovati.")


def save_table(table: pd.DataFrame, filepath: Path, index: bool = True) -> None:
    """
    Salva una tabella in CSV creando la cartella di destinazione se necessario.
    """
    filepath.parent.mkdir(parents=True, exist_ok=True)
    table.to_csv(filepath, index=index)
    print(f"Salvato CSV: {filepath}")


def save_figure(fig: plt.Figure, filepath: Path, dpi: int = 180) -> None:
    """
    Salva una figura matplotlib creando la cartella di destinazione se necessario.
    """
    filepath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(filepath, dpi=dpi, bbox_inches="tight")
    print(f"Salvato grafico: {filepath}")


ensure_project_directories()


def load_dataset(filepath: Path, file_format: str = "parquet") -> pd.DataFrame:
    """
    Carica un dataset da disco e stampa informazioni essenziali.
    """
    try:
        print(f"Caricamento del file: {filepath}")
        if file_format == "parquet":
            loaded_df = pd.read_parquet(filepath)
        elif file_format == "csv":
            loaded_df = pd.read_csv(filepath)
        else:
            raise ValueError("Formato file non supportato. Usare 'parquet' o 'csv'.")

        mem_usage = loaded_df.memory_usage(deep=True).sum() / (1024 ** 2)
        print("Dataset caricato correttamente.")
        print("-" * 50)
        print(f"Shape: {loaded_df.shape[0]} righe, {loaded_df.shape[1]} colonne")
        print(f"Memoria occupata: {mem_usage:.2f} MB")
        print("-" * 50)
        display(loaded_df.head(3))
        return loaded_df

    except Exception as exc:
        raise RuntimeError(f"Errore durante il caricamento del dataset: {exc}") from exc


# L'EDA viene svolta solo sul training set ufficiale.
train_df = load_dataset(TRAIN_PATH)
df = train_df.copy()

print(f"Test set definito ma non caricato in EDA: {TEST_PATH}")



### Analisi Generale e Tipi di Dato

Separiamo esplicitamente le **feature** dalle variabili target. Questa distinzione è necessaria per evitare che `label` o `attack_cat` vengano trattate come normali feature nelle statistiche descrittive, nelle correlazioni, nella PCA o nelle trasformazioni di preprocessing.


In [ ]:

# ==========================================
# 4. ANALISI GENERALE
# ==========================================
def analyze_data_types(df: pd.DataFrame, target_cols: List[str]) -> Dict[str, List[str]]:
    """
    Classifica le colonne del dataframe distinguendo feature e target.
    """
    existing_targets = [col for col in target_cols if col in df.columns]
    feature_cols = [col for col in df.columns if col not in existing_targets]

    feature_dict = {
        "features": feature_cols,
        "targets": existing_targets,
        "numeric": df[feature_cols].select_dtypes(include="number").columns.tolist(),
        "categorical": df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist(),
        "boolean": df[feature_cols].select_dtypes(include=["bool"]).columns.tolist(),
        "datetime": df[feature_cols].select_dtypes(include=["datetime64"]).columns.tolist(),
    }

    print("ANALISI DEI TIPI DI DATO")
    print("=" * 40)
    print(f"Target individuati: {', '.join(existing_targets) if existing_targets else 'nessuno'}")
    print(f"Feature totali: {len(feature_cols)}")

    for f_type in ["numeric", "categorical", "boolean", "datetime"]:
        f_list = feature_dict[f_type]
        print(f"Feature {f_type.capitalize()} ({len(f_list)}):")
        if f_list:
            print(f"   {', '.join(f_list[:10])}{'...' if len(f_list) > 10 else ''}")
        else:
            print("   Nessuna")

    return feature_dict


feature_types = analyze_data_types(df, TARGET_COLUMNS)
FEATURE_COLUMNS = feature_types["features"]
NUMERIC_FEATURES = feature_types["numeric"]
CATEGORICAL_FEATURES = feature_types["categorical"]


### Analisi dei Missing Values

Verifichiamo la presenza di valori mancanti espliciti (`NaN`) nel training set. Eventuali valori mancanti codificati semanticamente, come `-` nelle variabili categoriche, vengono analizzati nella sezione dedicata alla qualità dei dati.


In [ ]:
# ==========================================
# 6. ANALISI MISSING VALUES
# ==========================================
def analyze_missing_values(df: pd.DataFrame):
    """
    Genera report e visualizzazioni per i valori nulli.
    """
    missing_sum = df.isnull().sum()
    missing_pct = (missing_sum / len(df)) * 100
    missing_df = pd.DataFrame({'Total Missing': missing_sum, 'Percentage (%)': missing_pct})
    missing_df = missing_df[missing_df['Total Missing'] > 0].sort_values(by='Percentage (%)', ascending=False)

    print("🧩 ANALISI VALORI MANCANTI")
    print("=" * 40)

    if missing_df.empty:
        print("✅ Nessun valore mancante rilevato nel dataset standard (NaN).")
    else:
        display(missing_df)
        print("\nGenerazione matrice dei valori mancanti...")
        plt.figure(figsize=(10, 6))
        msno.matrix(df, figsize=(10, 5), sparkline=False, fontsize=10)
        plt.title("Matrice dei Valori Mancanti", fontsize=14)
        plt.show()

analyze_missing_values(df)

### Analisi e Gestione dei Duplicati

I duplicati vengono analizzati sul training set ufficiale prima delle analisi descrittive principali. In questo dataset sono presenti molti duplicati esatti; per evitare che pattern ripetuti influenzino eccessivamente la fase di apprendimento, le righe duplicate vengono rimosse dal training set mantenendo la prima occorrenza.

Questa operazione viene eseguita **solo sul training set**. Il test set resta separato e non viene usato per prendere decisioni durante l'EDA.


In [ ]:
# ==========================================
# 7. ANALISI E RIMOZIONE DUPLICATI
# ==========================================
def analyze_and_remove_duplicates(df: pd.DataFrame, output_dir: Path = EDA_DIRS["duplicates"]) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    """
    Analizza duplicati completi, salva i report e restituisce un dataframe deduplicato.
    """
    print("ANALISI DUPLICATI COMPLETI")
    print("=" * 45)

    duplicate_after_first_mask = df.duplicated(keep="first")
    duplicate_all_mask = df.duplicated(keep=False)

    n_rows = len(df)
    n_duplicate_after_first = int(duplicate_after_first_mask.sum())
    n_rows_in_duplicate_groups = int(duplicate_all_mask.sum())

    duplicates_after_first = df.loc[duplicate_after_first_mask]
    deduplicated_df = df.drop_duplicates(keep="first").reset_index(drop=True)

    duplicate_report = {
        "rows_before": n_rows,
        "duplicate_rows_after_first": n_duplicate_after_first,
        "rows_in_duplicate_groups": n_rows_in_duplicate_groups,
        "pct_duplicate_rows_after_first": 100 * n_duplicate_after_first / n_rows,
        "pct_rows_in_duplicate_groups": 100 * n_rows_in_duplicate_groups / n_rows,
        "rows_after": len(deduplicated_df),
        "rows_removed": n_rows - len(deduplicated_df),
    }
    save_table(pd.DataFrame([duplicate_report]), output_dir / "duplicate_summary.csv", index=False)

    print(f"Righe totali prima della deduplicazione: {n_rows}")
    print(f"Duplicati esclusa la prima occorrenza: {n_duplicate_after_first} ({duplicate_report['pct_duplicate_rows_after_first']:.2f}%)")
    print(f"Righe coinvolte in gruppi duplicati, prima occorrenza inclusa: {n_rows_in_duplicate_groups} ({duplicate_report['pct_rows_in_duplicate_groups']:.2f}%)")

    if TARGET_BINARY in df.columns:
        binary_dist = duplicates_after_first[TARGET_BINARY].value_counts(dropna=False).rename_axis(TARGET_BINARY).reset_index(name="conteggio")
        save_table(binary_dist, output_dir / "removed_duplicates_by_label.csv", index=False)
        if not binary_dist.empty:
            print("\nDistribuzione dei duplicati rimossi rispetto al target binario:")
            display(binary_dist.set_index(TARGET_BINARY))

    if TARGET_MULTICLASS in df.columns:
        multiclass_dist = duplicates_after_first[TARGET_MULTICLASS].value_counts(dropna=False).rename_axis(TARGET_MULTICLASS).reset_index(name="conteggio")
        save_table(multiclass_dist, output_dir / "removed_duplicates_by_attack_cat.csv", index=False)
        if not multiclass_dist.empty:
            print("\nDistribuzione dei duplicati rimossi rispetto alla categoria di attacco:")
            display(multiclass_dist.set_index(TARGET_MULTICLASS))

    print("\nDeduplicazione applicata al training set.")
    print(f"Righe dopo la deduplicazione: {len(deduplicated_df)}")
    print(f"Righe rimosse: {n_rows - len(deduplicated_df)}")

    return deduplicated_df, duplicate_report


df, duplicate_report = analyze_and_remove_duplicates(df)


### Statistiche Descrittive

Le statistiche descrittive vengono calcolate sul training set **deduplicato** e sulle sole feature numeriche, escludendo le variabili target. In questo modo `label` non viene interpretata come variabile continua e `attack_cat` resta disponibile per l'analisi del target.


In [ ]:
# ==========================================
# 5. STATISTICHE DESCRITTIVE
# ==========================================
def compute_descriptive_stats(df: pd.DataFrame, num_features: List[str], output_dir: Path = EDA_DIRS["descriptive_statistics"]) -> pd.DataFrame:
    """
    Calcola e salva statistiche descrittive per le feature numeriche.
    """
    if not num_features:
        print("Nessuna feature numerica disponibile per le statistiche descrittive.")
        empty_df = pd.DataFrame()
        save_table(empty_df, output_dir / "descriptive_statistics_numeric_features.csv", index=False)
        return empty_df

    print("STATISTICHE DESCRITTIVE DELLE FEATURE NUMERICHE")
    print("=" * 50)

    stats_df = df[num_features].describe().T
    stats_df["median"] = df[num_features].median()
    stats_df["skewness"] = df[num_features].skew()
    stats_df["kurtosis"] = df[num_features].kurtosis()
    stats_df["var"] = df[num_features].var()

    stats_to_display = stats_df[["mean", "median", "std", "var", "min", "max", "skewness", "kurtosis"]]
    display(stats_to_display)
    save_table(stats_to_display.reset_index().rename(columns={"index": "feature"}), output_dir / "descriptive_statistics_numeric_features.csv", index=False)

    highly_skewed_df = stats_df.loc[stats_df["skewness"].abs() > 3, ["skewness", "kurtosis"]].copy()
    highly_skewed_df = highly_skewed_df.sort_values("skewness", key=lambda s: s.abs(), ascending=False)
    save_table(highly_skewed_df.reset_index().rename(columns={"index": "feature"}), output_dir / "highly_skewed_features.csv", index=False)

    if not highly_skewed_df.empty:
        print("\nFeature con forte asimmetria assoluta (|skewness| > 3):")
        print(highly_skewed_df.index.tolist())
        print("Indicazione preliminare: valutare trasformazioni logaritmiche selettive o scaler robusti nella pipeline di preprocessing.")

    return stats_df


descriptive_stats = compute_descriptive_stats(df, NUMERIC_FEATURES)


### Analisi Outlier

Utilizziamo il metodo IQR (Interquartile Range) come diagnostica esplorativa sui valori estremi delle feature numeriche.

In un dataset NIDS gli outlier possono essere informativi e associati ad anomalie o attacchi; per questo non vengono rimossi automaticamente. Verranno eventualmente gestiti nella pipeline tramite trasformazioni selettive o scaling robusto.

L'analisi IQR evidenzia che diverse feature presentano un'elevata percentuale di valori classificati come outlier. In molti casi ciò è dovuto alla forte concentrazione della distribuzione su un singolo valore (IQR = 0), per cui qualsiasi valore differente viene identificato come outlier. Ad esempio, feature come ct_flw_http_mthd assumono nella maggior parte dei casi il valore 0 (assenza di traffico HTTP), mentre valori maggiori rappresentano situazioni meno frequenti ma non necessariamente anomale; analogamente, variabili come swin e dwin sono fortemente concentrate sul valore 255, per cui gli altri valori vengono identificati come outlier dal criterio IQR. Teniamo conto dell'esistenza di feature di questo tipo nelle decisioni successive.

In [ ]:
# ==========================================
# 8. ANALISI OUTLIER
# ==========================================
def detect_outliers(df: pd.DataFrame, num_features: List[str], output_dir: Path) -> pd.DataFrame:
    """
    Calcola gli outlier con metodo IQR sulle feature numeriche non binarie.
    Salva CSV e boxplot solo per le feature con almeno un outlier.
    """
    features_to_check = []
    for col in num_features:
        if col in ["id"]:
            continue
        if df[col].nunique(dropna=False) <= 2:
            continue
        features_to_check.append(col)

    outlier_rows = []
    for col in features_to_check:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        count = int(((df[col] < lower_bound) | (df[col] > upper_bound)).sum())
        outlier_rows.append({
            "feature": col,
            "outlier_count": count,
            "outlier_pct": 100 * count / len(df),
            "iqr_zero": iqr == 0,
            "q1": q1,
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
        })

    outlier_df = pd.DataFrame(outlier_rows).sort_values("outlier_pct", ascending=False)
    outlier_df = outlier_df[outlier_df["outlier_count"] > 0].reset_index(drop=True)
    save_table(outlier_df, output_dir / "outlier_summary_features_with_outliers.csv", index=False)

    print("ANALISI OUTLIER (METODO IQR)")
    print("=" * 45)
    display(outlier_df.head(15))
    print("Interpretazione: gli outlier sono trattati come informazione potenzialmente utile per la rilevazione di attacchi, non come rumore da eliminare automaticamente.")

    plot_features = outlier_df.loc[outlier_df["outlier_count"] > 0, "feature"].tolist()
    boxplot_dir = EDA_DIRS["outlier_boxplots"]
    for col in plot_features:
        fig, ax = plt.subplots(figsize=(5, 4))
        sns.boxplot(y=df[col], ax=ax, color="coral")
        ax.set_title(f"Boxplot outlier: {col}")
        ax.set_yscale("symlog", linthresh=1)
        save_figure(fig, boxplot_dir / f"{slugify(col)}_boxplot.png")
        plt.close(fig)

    display_features = outlier_df.loc[outlier_df["outlier_count"] > 0, "feature"].head(15).tolist()
    if display_features:
        ncols = 5
        nrows = int(np.ceil(len(display_features) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows))
        axes = np.array(axes).reshape(-1)

        for ax, col in zip(axes, display_features):
            sns.boxplot(y=df[col], ax=ax, color="coral")
            ax.set_title(col)
            ax.set_yscale("symlog", linthresh=1)

        for ax in axes[len(display_features):]:
            ax.remove()

        plt.tight_layout()
        save_figure(fig, output_dir / "top15_outlier_boxplots_grid.png")
        plt.show()

    return outlier_df


outlier_summary = detect_outliers(df, NUMERIC_FEATURES, EDA_DIRS["outliers"])


### Distribuzioni delle Feature

Visualizziamo come si comportano i dati.  
Per i log di rete, ci aspettiamo distribuzioni fortemente asimmetriche a legge di potenza.

In [ ]:
# ==========================================
# 9. DISTRIBUZIONI
# ==========================================
def transform_for_distribution_plot(series: pd.Series) -> Tuple[pd.Series, str]:
    """
    Trasforma una feature numerica per rendere leggibili distribuzioni fortemente asimmetriche.
    """
    clean_series = pd.to_numeric(series, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    if clean_series.empty:
        return clean_series, series.name

    if (clean_series >= 0).all():
        return np.log1p(clean_series), f"log1p({series.name})"

    signed_log_series = np.sign(clean_series) * np.log1p(np.abs(clean_series))
    return signed_log_series, f"signed_log1p({series.name})"


def plot_distributions(df: pd.DataFrame, feature_dict: Dict[str, List[str]], max_plots: int = 5) -> None:
    """
    Mostra un sottoinsieme di distribuzioni nel notebook e salva su Drive i grafici per tutte le feature.
    """
    print("ANALISI DELLE DISTRIBUZIONI")

    num_feats_all = [f for f in feature_dict["numeric"] if f != "id"]
    num_feats_display = num_feats_all[:max_plots]

    for col in num_feats_all:
        plot_values, plot_label = transform_for_distribution_plot(df[col])
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(plot_values, kde=True, ax=axes[0], color="skyblue", bins=50)
        axes[0].set_title(f"Istogramma: {col}")
        axes[0].set_xlabel(plot_label)
        axes[0].set_yscale("log")

        sns.violinplot(x=plot_values, ax=axes[1], color="lightgreen")
        axes[1].set_title(f"Violin plot: {col}")
        axes[1].set_xlabel(plot_label)
        plt.tight_layout()
        save_figure(fig, EDA_DIRS["numeric_distributions"] / f"{slugify(col)}_distribution.png")
        plt.close(fig)

    if num_feats_display:
        fig, axes = plt.subplots(len(num_feats_display), 2, figsize=(14, 4 * len(num_feats_display)))
        if len(num_feats_display) == 1:
            axes = np.array([axes])

        for i, col in enumerate(num_feats_display):
            plot_values, plot_label = transform_for_distribution_plot(df[col])
            sns.histplot(plot_values, kde=True, ax=axes[i, 0], color="skyblue", bins=50)
            axes[i, 0].set_title(f"Istogramma: {col}")
            axes[i, 0].set_xlabel(plot_label)
            axes[i, 0].set_yscale("log")

            sns.violinplot(x=plot_values, ax=axes[i, 1], color="lightgreen")
            axes[i, 1].set_title(f"Violin plot: {col}")
            axes[i, 1].set_xlabel(plot_label)
        plt.tight_layout()
        plt.show()

    cat_feats_all = feature_dict["categorical"]
    cat_feats_display = cat_feats_all[:max_plots]

    for col in cat_feats_all:
        top_cats = df[col].value_counts().nlargest(30).index
        fig, ax = plt.subplots(figsize=(8, max(4, 0.35 * len(top_cats))))
        sns.countplot(data=df[df[col].isin(top_cats)], y=col, ax=ax, order=top_cats)
        ax.set_title(f"Top categorie: {col}")
        ax.set_xlabel("conteggio")
        plt.tight_layout()
        save_figure(fig, EDA_DIRS["categorical_distributions"] / f"{slugify(col)}_countplot.png")
        plt.close(fig)

    if cat_feats_display:
        fig, axes = plt.subplots(1, len(cat_feats_display), figsize=(5 * len(cat_feats_display), 5))
        if len(cat_feats_display) == 1:
            axes = [axes]

        for i, col in enumerate(cat_feats_display):
            top_cats = df[col].value_counts().nlargest(10).index
            sns.countplot(data=df[df[col].isin(top_cats)], y=col, ax=axes[i], order=top_cats)
            axes[i].set_title(f"Top 10 categorie: {col}")
        plt.tight_layout()
        plt.show()


plot_distributions(df, feature_types)


### Analisi delle Correlazioni

Calcoliamo il coefficiente di Spearman (che valuta la monotonicità senza assumere linearità) per individuare multicollinearità tra le feature.  

Feature altamente correlate (es. sbytes e spkts) introducono rumore nei modelli lineari.

In [ ]:
# ==========================================
# 10. CORRELAZIONI (SPEARMAN)
# ==========================================
def analyze_correlations(df: pd.DataFrame, num_features: List[str], threshold: float = 0.85, output_dir: Path = EDA_DIRS["correlations"]) -> pd.DataFrame:
    """
    Calcola correlazioni Spearman tra feature numeriche, salva tabella delle coppie fortemente correlate e heatmap.
    """
    features = [f for f in num_features if f != "id"]
    if not features:
        print("Nessuna feature numerica disponibile per l'analisi delle correlazioni.")
        empty_df = pd.DataFrame(columns=["feature_1", "feature_2", "spearman_corr"])
        save_table(empty_df, output_dir / "high_spearman_correlations.csv", index=False)
        return empty_df

    print("ANALISI CORRELAZIONI TRA FEATURE NUMERICHE")
    print("=" * 50)

    corr_matrix = df[features].corr(method="spearman")
    corr_matrix.to_csv(output_dir / "spearman_correlation_matrix.csv")
    print(f"Salvato CSV: {output_dir / 'spearman_correlation_matrix.csv'}")

    high_corr_pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            corr_value = corr_matrix.iloc[i, j]
            if abs(corr_value) > threshold:
                high_corr_pairs.append({
                    "feature_1": corr_matrix.columns[i],
                    "feature_2": corr_matrix.columns[j],
                    "spearman_corr": corr_value,
                })

    high_corr_df = pd.DataFrame(high_corr_pairs)
    if high_corr_df.empty:
        print(f"Nessuna coppia con |correlazione| > {threshold}.")
        high_corr_df = pd.DataFrame(columns=["feature_1", "feature_2", "spearman_corr"])
    else:
        high_corr_df = high_corr_df.sort_values("spearman_corr", key=lambda s: s.abs(), ascending=False)
        print(f"Coppie con |correlazione| > {threshold}:")
        display(high_corr_df)
        print("Indicazione: valutare feature selection o modelli robusti alla multicollinearita'; non rimuovere feature in modo automatico solo sulla base della correlazione.")

    save_table(high_corr_df, output_dir / "high_spearman_correlations.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, mask=mask, cmap="coolwarm", annot=False, vmin=-1, vmax=1, ax=ax)
    ax.set_title("Matrice di Correlazione Spearman - Feature Numeriche")
    save_figure(fig, output_dir / "spearman_correlation_heatmap.png")
    plt.show()

    return high_corr_df


high_corr_features = analyze_correlations(df, NUMERIC_FEATURES)


### Analisi della Variabile Target

Analizziamo sia il target binario `label` sia il target multiclasse `attack_cat`. Questa sezione serve a quantificare lo sbilanciamento delle classi e a motivare la scelta delle metriche di valutazione nelle fasi successive.


In [ ]:
# ==========================================
# 11. ANALISI VARIABILE TARGET
# ==========================================
def analyze_target(df: pd.DataFrame, target_cols: List[str], output_dir: Path = EDA_DIRS["target"]) -> Dict[str, pd.DataFrame]:
    """
    Analizza il bilanciamento delle variabili target disponibili e salva grafici/CSV.
    """
    print("ANALISI VARIABILI TARGET")
    print("=" * 40)

    available_targets = [col for col in target_cols if col in df.columns]
    if not available_targets:
        print("Nessuna variabile target trovata.")
        return {}

    target_distributions = {}
    fig, axes = plt.subplots(1, len(available_targets), figsize=(6 * len(available_targets), 5))
    if len(available_targets) == 1:
        axes = [axes]

    for ax, target in zip(axes, available_targets):
        counts = df[target].value_counts(dropna=False)
        pcts = df[target].value_counts(normalize=True, dropna=False) * 100
        distribution_df = pd.DataFrame({"conteggio": counts, "percentuale": pcts.round(2)})
        target_distributions[target] = distribution_df

        distribution_to_save = distribution_df.reset_index()
        distribution_to_save = distribution_to_save.rename(columns={distribution_to_save.columns[0]: target})
        save_table(distribution_to_save, output_dir / f"{slugify(target)}_distribution.csv", index=False)

        print(f"\nDistribuzione target: {target}")
        display(distribution_df)

        sns.countplot(data=df, y=target, order=counts.index, ax=ax, palette="magma")
        ax.set_title(f"Distribuzione: {target}")
        ax.set_xlabel("conteggio")
        ax.set_ylabel(target)

        single_fig, single_ax = plt.subplots(figsize=(7, 5))
        sns.countplot(data=df, y=target, order=counts.index, ax=single_ax, palette="magma")
        single_ax.set_title(f"Distribuzione: {target}")
        single_ax.set_xlabel("conteggio")
        single_ax.set_ylabel(target)
        plt.tight_layout()
        save_figure(single_fig, output_dir / f"{slugify(target)}_distribution.png")
        plt.close(single_fig)

    plt.tight_layout()
    save_figure(fig, output_dir / "target_distributions_combined.png")
    plt.show()

    return target_distributions


target_distributions = analyze_target(df, TARGET_COLUMNS)


### Relazioni tra Feature

Esploriamo alcune coppie di feature numeriche rilevanti per il dominio NIDS. L'obiettivo è osservare se traffico normale e attacchi mostrano separazioni visive in spazi a due dimensioni.


In [ ]:
# ==========================================
# 12. RELAZIONI TRA FEATURE
# ==========================================
def plot_relationships(df: pd.DataFrame, feature_pairs: List[Tuple[str, str]], target: str = TARGET_BINARY, sample_size: int = 20000, output_dir: Path = EDA_DIRS["feature_relationships"]) -> None:
    """
    Visualizza e salva coppie di feature selezionate rispetto al target binario.
    """
    if target not in df.columns:
        print(f"Target '{target}' non disponibile.")
        return

    available_pairs = [(x, y) for x, y in feature_pairs if x in df.columns and y in df.columns]
    if not available_pairs:
        print("Nessuna coppia di feature disponibile per lo scatter plot.")
        return

    plot_df = df.sample(n=min(sample_size, len(df)), random_state=RANDOM_SEED)

    for x_col, y_col in available_pairs:
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue=target, alpha=0.35, s=15, ax=ax, palette="Set1")
        ax.set_title(f"{x_col} vs {y_col}")
        ax.set_xscale("symlog", linthresh=1)
        ax.set_yscale("symlog", linthresh=1)
        plt.tight_layout()
        save_figure(fig, output_dir / f"{slugify(x_col)}_vs_{slugify(y_col)}.png")
        plt.close(fig)

    fig, axes = plt.subplots(1, len(available_pairs), figsize=(6 * len(available_pairs), 5))
    if len(available_pairs) == 1:
        axes = [axes]

    for ax, (x_col, y_col) in zip(axes, available_pairs):
        sns.scatterplot(data=plot_df, x=x_col, y=y_col, hue=target, alpha=0.35, s=15, ax=ax, palette="Set1")
        ax.set_title(f"{x_col} vs {y_col}")
        ax.set_xscale("symlog", linthresh=1)
        ax.set_yscale("symlog", linthresh=1)

    plt.tight_layout()
    save_figure(fig, output_dir / "selected_feature_relationships_combined.png")
    plt.show()


selected_feature_pairs = [("sbytes", "dbytes"), ("sload", "dload"), ("dur", "rate")]
plot_relationships(df, selected_feature_pairs)


### Analisi della Qualità dei Dati (Inconsistenze)

Indaghiamo la presenza di valori semanticamente invalidi o errori di codifica.  
Nel UNSW-NB15, ad esempio, "-" nei servizi indica un dato mancante.

In [ ]:
# ==========================================
# 13. QUALITA' DEI DATI
# ==========================================
def check_data_quality(df: pd.DataFrame, cat_features: List[str], output_dir: Path = EDA_DIRS["data_quality"]) -> pd.DataFrame:
    """
    Ricerca valori categorici codificati come '-' e salva il report in CSV.
    """
    print("CONTROLLO QUALITA' DEI DATI")
    print("=" * 40)

    rows = []
    for col in cat_features:
        dash_mask = df[col].astype(str).eq("-")
        dash_count = int(dash_mask.sum())
        if dash_count > 0:
            rows.append({
                "feature": col,
                "value": "-",
                "count": dash_count,
                "percentage": 100 * dash_count / len(df),
                "recommended_action": "codificare come Unknown nella pipeline",
            })

    quality_df = pd.DataFrame(rows, columns=["feature", "value", "count", "percentage", "recommended_action"])
    save_table(quality_df, output_dir / "categorical_inconsistencies.csv", index=False)

    if quality_df.empty:
        print("Nessuna inconsistenza semantica evidente rilevata nelle feature categoriche.")
    else:
        display(quality_df)

    return quality_df


quality_issues = check_data_quality(df, CATEGORICAL_FEATURES)


### Preparazione Dati (Blueprint di Preprocessing)

Questa sezione non applica trasformazioni distruttive al dataframe dell'EDA. Definisce invece un blueprint compatibile con una futura pipeline `scikit-learn`, così preprocessing e modello potranno essere stimati dentro cross-validation senza data leakage.


In [ ]:
# ==========================================
# 14. PREPARAZIONE DATI (BLUEPRINT)
# ==========================================
def replace_service_dash(X: pd.DataFrame) -> pd.DataFrame:
    """
    Sostituisce '-' con 'Unknown' nella feature service, preservando le altre colonne.
    """
    X = X.copy()
    if "service" in X.columns:
        X["service"] = X["service"].replace("-", "Unknown")
    return X


def build_preprocessing_pipeline(numeric_features: List[str], categorical_features: List[str]) -> ColumnTransformer:
    """
    Costruisce un preprocessing sklearn da usare nella fase di modellazione.
    """
    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("dash_cleaner", FunctionTransformer(replace_service_dash, validate=False)),
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_features),
            ("categorical", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
    )

    return preprocessor


preprocessing_blueprint = build_preprocessing_pipeline(NUMERIC_FEATURES, CATEGORICAL_FEATURES)
print("Blueprint di preprocessing definito per la futura fase di modellazione.")
print("Nota: il blueprint dovrà essere inserito in una Pipeline sklearn insieme al modello e validato con cross-validation.")


### Analisi Avanzate: PCA e Feature Importance Esplorativa

PCA e Random Forest vengono usate qui solo come strumenti esplorativi sul training set deduplicato. Non rappresentano una valutazione predittiva finale: le performance dei modelli saranno valutate successivamente con cross-validation e test set finale.


In [ ]:
# ==========================================
# 15. ANALISI AVANZATE ESPLORATIVE
# ==========================================
def run_advanced_analysis(df: pd.DataFrame, num_features: List[str], target: str = TARGET_BINARY, sample_size: int = 30000, output_dir: Path = EDA_DIRS["pca_feature_importance"]) -> pd.DataFrame:
    """
    Esegue PCA esplorativa e feature importance preliminare su feature numeriche, salvando risultati e grafici.
    """
    features = [f for f in num_features if f != "id"]
    if not features or target not in df.columns:
        print("Feature numeriche o target non disponibili per l'analisi avanzata.")
        empty_df = pd.DataFrame(columns=["feature", "importance"])
        save_table(empty_df, output_dir / "random_forest_feature_importance.csv", index=False)
        return empty_df

    analysis_df = df.sample(n=min(sample_size, len(df)), random_state=RANDOM_SEED)
    X = analysis_df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
    y = analysis_df[target]

    print("ANALISI AVANZATE ESPLORATIVE")
    print("=" * 45)
    print(f"Campione usato per analisi esplorativa: {len(analysis_df)} righe")

    X_scaled = StandardScaler().fit_transform(X)
    pca = PCA(n_components=2, random_state=RANDOM_SEED)
    components = pca.fit_transform(X_scaled)

    pca_components_df = pd.DataFrame({
        "pc1": components[:, 0],
        "pc2": components[:, 1],
        target: y.to_numpy(),
    })
    pca_variance_df = pd.DataFrame({
        "component": ["PC1", "PC2"],
        "explained_variance_ratio": pca.explained_variance_ratio_,
    })
    save_table(pca_components_df, output_dir / "pca_components_sample.csv", index=False)
    save_table(pca_variance_df, output_dir / "pca_explained_variance.csv", index=False)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.scatterplot(x=components[:, 0], y=components[:, 1], hue=y, alpha=0.45, s=15, palette="coolwarm", ax=ax)
    ax.set_title(f"PCA esplorativa - varianza spiegata: {pca.explained_variance_ratio_.sum():.2f}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    save_figure(fig, output_dir / "pca_scatter.png")
    plt.show()

    rf = RandomForestClassifier(
        n_estimators=100,
        max_depth=8,
        class_weight="balanced",
        random_state=RANDOM_SEED,
        n_jobs=-1,
    )
    rf.fit(X, y)

    importance_df = pd.DataFrame({"feature": features, "importance": rf.feature_importances_})
    importance_df = importance_df.sort_values("importance", ascending=False)
    save_table(importance_df, output_dir / "random_forest_feature_importance.csv", index=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=importance_df.head(10), x="importance", y="feature", palette="crest", ax=ax)
    ax.set_title("Top 10 feature importance esplorativa - Random Forest")
    save_figure(fig, output_dir / "random_forest_top10_feature_importance.png")
    plt.show()

    print("Nota: questa feature importance e' descrittiva e non sostituisce la valutazione dei modelli.")
    return importance_df


exploratory_feature_importance = run_advanced_analysis(df, NUMERIC_FEATURES)


### Sintesi Operativa dell'EDA

| Evidenza osservata | Decisione per le fasi successive |
|---|---|
| Il dataset fornisce split ufficiali training/test | EDA e model selection solo sul training; test set usato solo per valutazione finale |
| Presenza elevata di duplicati esatti nel training set | Rimozione dei duplicati dal training mantenendo la prima occorrenza |
| Target binario sbilanciato e classi `attack_cat` molto sbilanciate | Uso di metriche oltre accuracy: precision, recall, F1, ROC-AUC; per multiclasse preferire macro-F1 |
| Feature numeriche fortemente asimmetriche e con outlier | Non rimuovere automaticamente outlier; valutare RobustScaler e trasformazioni selettive nella pipeline |
| Feature categorica `service` contiene valori `-` | Codificare `-` come `Unknown` nel preprocessing |
| Correlazioni forti tra diverse feature numeriche | Valutare feature selection/modelli robusti, senza rimozione automatica basata solo sulla correlazione |
| PCA e feature importance esplorativa sono analisi descrittive | Performance e selezione modelli saranno valutate con cross-validation sul training set |

Nonostante la proporzione finale tra training e test set differisca dalla comune euristica 80/20 a seguito della deduplicazione del training set, è stata mantenuta la partizione ufficiale del dataset per garantire una valutazione imparziale e confrontabile con la letteratura, preservando il ruolo del test set come benchmark finale.
